# Neutron imaging: Bragg edge division

This notebook goes with Question 11 of the simulation quiz *Bragg Edge Imaging on Viking Sword*.

The attenuation of iron (Fe_alpha) changes abruptly at its Bragg edges, while the attenuation of rust (FeOOH) is almost constant with wavelength. If we take one radiogram just below and one just above a Bragg edge of iron and divide them, the iron changes a lot between the two images whereas the rust hardly changes. The logarithm of the ratio gives a very high contrast between iron and rust:

$$C(x,y) = \ln\frac{I_{\lambda_{\mathrm{long}}}(x,y)}{I_{\lambda_{\mathrm{short}}}(x,y)}$$

As explained in the quiz, we skip the division by the open beam image here to save time.

**How to use it**

1. Run the Sword_ODIN simulation at the short wavelength and at the long wavelength found in Question 9 (same chopper mode and geometry, `Sample=1`, at least 1E7 neutron rays).
2. From each simulation folder, copy `absorption_picture.dat` into the folder `my_data` next to this notebook, and rename them `data_short.dat` and `data_long.dat`.
3. Set the two wavelengths below and run all the cells.

Both wavelengths must be inside the range of the chopper mode you use: chopper_mode=3 starts at 3.8 Å, so for 3.7 Å and 4.3 Å use chopper_mode=2 (2.5-6.5 Å) for both radiograms.

If `my_data/data_short.dat` is not found, the notebook uses the example files in `example_data` (simulated at 3.7 Å and 4.3 Å, chopper_mode=2, 1E7 neutron rays).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

short_wavelength = 3.7  # AA, only used for the plot titles
long_wavelength = 4.3   # AA


def read_mccode_2d(path):
    """Read a 2D McStas monitor file (McCode text format): returns intensity, error, counts and extent [cm]."""
    header, rows = {}, []
    with open(path) as f:
        for line in f:
            if line.startswith("#"):
                key, _, value = line[1:].partition(":")
                header.setdefault(key.strip(), value.strip())
            elif line.strip():
                rows.append([float(v) for v in line.split()])
    ny = int(header["type"].split("(")[1].split(",")[1].rstrip(")"))
    data = np.array(rows)
    intensity, error, counts = data[:ny], data[ny:2 * ny], data[2 * ny:3 * ny]
    extent = [float(v) for v in header["xylimits"].split()]
    return intensity, error, counts, extent, header


if os.path.exists(os.path.join("my_data", "data_short.dat")):
    short_file, long_file = os.path.join("my_data", "data_short.dat"), os.path.join("my_data", "data_long.dat")
else:
    short_file = os.path.join("example_data", "bragg_edge_3.7A.dat")
    long_file = os.path.join("example_data", "bragg_edge_4.3A.dat")
print("Using", short_file, "and", long_file)

short, _, _, extent, _ = read_mccode_2d(short_file)
long_, _, _, _, _ = read_mccode_2d(long_file)

## The two radiograms

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, img, lam in zip(axes, [short, long_], [short_wavelength, long_wavelength]):
    im = ax.imshow(img, origin="lower", extent=extent, cmap="viridis")
    ax.set_title(f"Radiogram at {lam} Å")
    ax.set_xlabel("X position [cm]")
    ax.set_ylabel("Y position [cm]")
    fig.colorbar(im, ax=ax, label="Intensity [n/s]")
plt.tight_layout()
plt.show()

## Division and logarithm

In [ ]:
with np.errstate(divide="ignore", invalid="ignore"):
    ratio = np.where((short > 0) & (long_ > 0), long_ / short, np.nan)
    log_ratio = np.log(ratio)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, img, title in zip(axes, [ratio, log_ratio],
                          [f"Ratio {long_wavelength} Å / {short_wavelength} Å",
                           f"ln( {long_wavelength} Å / {short_wavelength} Å )"]):
    low, high = np.nanpercentile(img, [1, 99])
    im = ax.imshow(img, origin="lower", extent=extent, cmap="viridis", vmin=low, vmax=high)
    ax.set_title(title)
    ax.set_xlabel("X position [cm]")
    ax.set_ylabel("Y position [cm]")
    fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

Where the beam only goes through iron, the ratio differs clearly from 1. Where there is rust, the attenuation hardly changes between the two wavelengths. Look for the areas that stand out in the logarithm image: this is where the rust is. Use them to place the markers in Question 11 of the quiz.